## **GET METADATA FROM DATAHUB AND CONVERT INTO JSONL**

In [6]:
# 01_export_columns.py
import os
import json
import requests
import re
from typing import Dict, List, Any, Optional, Tuple
from dotenv import load_dotenv, find_dotenv
from datetime import datetime, timezone, timedelta  # <-- for time formatting

load_dotenv(find_dotenv(), override=True)

# ===== Config =====
GMS_BASE = os.getenv("DATAHUB_GMS_URL", "http://datahub-gms:8080")
GRAPHQL_URL = os.getenv("DATAHUB_GRAPHQL_URL", f"{GMS_BASE}/api/graphql")
TOKEN = os.getenv("DATAHUB_TOKEN")
if not TOKEN:
    raise RuntimeError("DATAHUB_TOKEN is not set.")
PAGE_SIZE = int(os.getenv("DATAHUB_PAGE_SIZE", "100"))
TIMEOUT = 60
ACTOR = os.getenv("DATAHUB_ACTOR")  # optional, e.g. urn:li:corpuser:datahub

# Output directory for per-database JSONL files (folder, not a glob)
OUTPUT_DIR = os.getenv("EXPORT_JSONL_OUTPUT", "./exports")

# Platforms using database.schema.table
RELATIONAL_3LEVEL = {"postgres", "snowflake", "redshift", "bigquery", "mssql", "mysql"}

# ===== GraphQL =====
SCROLL_QUERY = """
query scrollPG($types: [EntityType!], $query: String!, $count: Int!, $scrollId: String) {
  scrollAcrossEntities(input: { types: $types, query: $query, count: $count, scrollId: $scrollId }) {
    nextScrollId
    searchResults {
      entity {
        urn
        type
        ... on Dataset { urn }
      }
    }
  }
}
""".strip()

DATASET_META_QUERY = """
query datasetMeta($urn: String!) {
  dataset(urn: $urn) {
    urn
    platform { name properties { datasetNameDelimiter } }
    parentContainers {
      containers {
        properties { name description }
        editableProperties { description }   # pick up UI/GraphQL-edited docs
      }
    }
    schemaMetadata(version: 0) {
      fields { fieldPath description nativeDataType type }
    }
  }
}
""".strip()

# Best-effort: some versions expose dataset.lastIngested; if not, we return None.
DATASET_LAST_INGESTED_QUERY = """
query datasetSys($urn: String!) {
  dataset(urn: $urn) { lastIngested }
}
""".strip()

# Profile query that matches your Avro (uniqueCount, stdev, stats as strings)
DATASET_PROFILE_QUERY = """
query datasetProfile($urn: String!) {
  dataset(urn: $urn) {
    datasetProfiles(limit: 1) {
      rowCount
      fieldProfiles {
        fieldPath
        uniqueCount
        uniqueProportion
        nullCount
        nullProportion
        min
        max
        mean
        median
        stdev
        sampleValues
      }
    }
  }
}
""".strip()

def _graphql(query: str, variables: dict) -> dict:
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {TOKEN}"}
    if ACTOR:
        headers["X-DataHub-Actor"] = ACTOR  # optional but helpful on some deployments
    resp = requests.post(GRAPHQL_URL, json={"query": query, "variables": variables},
                         headers=headers, timeout=TIMEOUT)
    resp.raise_for_status()
    data = resp.json()
    if data.get("errors"):
        # bubble up server errors for easier debugging
        raise RuntimeError(data["errors"][0])
    return data["data"]

def iter_dataset_urns():
    scroll_id = None
    while True:
        data = _graphql(
            SCROLL_QUERY,
            {"types": ["DATASET"], "query": "*", "count": PAGE_SIZE, "scrollId": scroll_id},
        )["scrollAcrossEntities"]
        for item in data.get("searchResults") or []:
            ent = item.get("entity") or {}
            urn = ent.get("urn")
            if urn:
                yield urn
        scroll_id = data.get("nextScrollId")
        if not scroll_id:
            break

def _parse_urn_name(urn: str) -> Optional[str]:
    # urn:li:dataset:(platformUrn,name,origin) -> name
    try:
        inner = urn[urn.index("(")+1 : urn.rindex(")")]
        _, name_part, _ = inner.split(",", 2)
        return name_part
    except Exception:
        return None

def _split_from_urn(urn: str, platform_name: Optional[str], delim: str = ".") -> Tuple[Optional[str], Optional[str], Optional[str], Optional[str]]:
    # Returns (full_name, database, schema, table) from URN
    name_part = _parse_urn_name(urn)
    if not name_part:
        return None, None, None, None
    plat = (platform_name or "").lower()
    if plat in RELATIONAL_3LEVEL:
        parts = name_part.split(delim)
        if len(parts) >= 3:
            return name_part, parts[0], parts[1], delim.join(parts[2:])
        elif len(parts) == 2:
            return name_part, parts[0], None, parts[1]
        else:
            return name_part, None, None, parts[0]
    return name_part, None, None, name_part

def _best_effort_last_ingested(urn: str) -> Optional[int]:
    """Return dataset.lastIngested (ms epoch) if GraphQL supports it; else None."""
    try:
        d = _graphql(DATASET_LAST_INGESTED_QUERY, {"urn": urn})
        ds = d.get("dataset")
        if ds is not None:
            return ds.get("lastIngested")
    except Exception:
        pass
    return None

# ---- time formatting helpers (Jakarta local without timezone suffix) ----
_JAKARTA = timezone(timedelta(hours=7))
def _ms_to_local_naive(ms: Optional[int]) -> Optional[str]:
    """
    Convert epoch ms to Asia/Jakarta local time and return ISO-like string WITHOUT timezone.
    Example: 2025-09-22T15:00:00
    """
    if ms is None:
        return None
    dt_local = datetime.fromtimestamp(ms / 1000, tz=timezone.utc).astimezone(_JAKARTA)
    return dt_local.strftime("%Y-%m-%dT%H:%M:%S")

# ---- profiling helpers ----
_NUMERIC_KEYWORDS = (
    "int", "integer", "smallint", "bigint", "tinyint",
    "double", "float", "real",
    "decimal", "numeric", "number", "money"
)
def _is_numeric(datatype: Optional[str]) -> bool:
    if not datatype:
        return False
    s = str(datatype).lower()
    return any(k in s for k in _NUMERIC_KEYWORDS)

def _norm_field(s: Optional[str]) -> Optional[str]:
    if s is None:
        return None
    # strip backticks/quotes/brackets and lowercase (common profile/schema diffs)
    return s.strip().strip('`"[]').lower()

def _as_float(x):
    """Coerce numbers that arrive as strings to float; leave None if not parseable."""
    if x is None:
        return None
    try:
        return float(x)
    except Exception:
        return None

def _pick_sample_value(fp: dict) -> str:
    sv = fp.get("sampleValues") or []
    if isinstance(sv, list) and sv:
        return str(sv[0])
    return ""

def _profile_map_by_field(urn: str) -> dict:
    """
    Returns: { normalized_field -> { min,max,mean,stddev, nullCount, distinct_count, sample_value } }
    All numeric stats are floats (or None); sample_value is a short string.
    """
    try:
        d = _graphql(DATASET_PROFILE_QUERY, {"urn": urn})
    except Exception as e:
        print(f"[profile] GraphQL error for {urn}: {e}")
        return {}

    ds = (d.get("dataset") or {})
    prof = (ds.get("datasetProfiles") or [None])[0]
    if not prof:
        return {}

    out = {}
    for fp in (prof.get("fieldProfiles") or []):
        key = _norm_field(fp.get("fieldPath"))
        if not key:
            continue

        # Your Avro: min/max/mean/median/stdev are strings -> coerce to floats.
        min_v   = _as_float(fp.get("min"))
        max_v   = _as_float(fp.get("max"))
        mean_v  = _as_float(fp.get("mean"))
        stdev_v = _as_float(fp.get("stdev"))

        out[key] = {
            "nullCount": fp.get("nullCount"),
            "min": min_v,
            "max": max_v,
            "mean": mean_v,
            "stddev": stdev_v,
            "distinct_count": fp.get("uniqueCount"),  # UI “distinct” == Avro uniqueCount
            "sample_value": _pick_sample_value(fp),
        }
    return out

def fetch_dataset_columns(urn: str) -> List[Dict[str, Any]]:
    """
    Returns a list of column dicts:
      {
        database, table, column, datatype, description,
        database_description, last_ingested,
        min, max, mean, stddev, sample_value,
        (optional) distinct_count
      }
      - Numeric columns: min/max/mean/stddev are numbers (or null if absent)
      - Non-numeric: those four fields = "unknown"
      - sample_value: one example value if present, else ""
    """
    d = _graphql(DATASET_META_QUERY, {"urn": urn}).get("dataset")
    if not d:
        return []

    platform = d.get("platform") or {}
    platform_name = (platform.get("name") or "").lower()
    delim = (platform.get("properties") or {}).get("datasetNameDelimiter") or "."

    _, db, schema, table = _split_from_urn(urn, platform_name, delim)

    # Database description from parent containers
    db_desc = ""
    containers = ((d.get("parentContainers") or {}).get("containers") or [])
    for c in containers:
        props = c.get("properties") or {}
        edit  = c.get("editableProperties") or {}
        if (props.get("name") or "").lower() == (db or "").lower():
            db_desc = (edit.get("description") or props.get("description") or "").strip()
            break
    if not db_desc:
        for c in containers:
            props = c.get("properties") or {}
            edit  = c.get("editableProperties") or {}
            cand = (edit.get("description") or props.get("description") or "").strip()
            if cand:
                db_desc = cand
                break

    # System recency: convert to Asia/Jakarta local string (no tz suffix)
    last_ingested_ms = _best_effort_last_ingested(urn)
    last_ingested_str = _ms_to_local_naive(last_ingested_ms)  # e.g., "2025-09-22T15:00:00"

    # Latest profile for this dataset (if exists)
    profile_map = _profile_map_by_field(urn)

    # MySQL: schema concept == database; omit "schema" entirely
    omit_schema = (platform_name == "mysql")

    rows: List[Dict[str, Any]] = []
    sm = d.get("schemaMetadata") or {}
    for f in (sm.get("fields") or []):
        field_path = f.get("fieldPath")
        native = f.get("nativeDataType") or (f.get("type") or "")
        is_num = _is_numeric(native)

        prof = profile_map.get(_norm_field(field_path), {}) if profile_map else {}
        sample_out = prof.get("sample_value", "")

        if is_num:
            min_v  = prof.get("min")
            max_v  = prof.get("max")
            mean_v = prof.get("mean")
            std_v  = prof.get("stddev")
        else:
            min_v = max_v = mean_v = std_v = "unknown"

        row: Dict[str, Any] = {
            "database": db,
            "table": table,
            "column": field_path,
            "datatype": native,
            "description": f.get("description") or "",
            "database_description": db_desc,
            "last_ingested": last_ingested_str,
            "min": min_v,
            "max": max_v,
            "mean": mean_v,
            "stddev": std_v,
            "sample_value": sample_out,
        }
        if (schema is not None) and (not omit_schema):
            row["schema"] = schema

        # OPTIONAL: include distinct count when present
        if prof.get("distinct_count") is not None:
            row["distinct_count"] = prof["distinct_count"]

        rows.append(row)
    return rows

def _sanitize_filename(name: str) -> str:
    """
    Make a safe filename from database name.
    - Lowercase
    - Replace non-alphanum with underscores
    - Collapse multiple underscores
    """
    if not name:
        return "unknown"
    s = name.strip().lower()
    s = re.sub(r"[^a-z0-9\\-_.]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "unknown"

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Keep one open file handle per database
    writers: Dict[str, Any] = {}
    counts: Dict[str, int] = {}

    def get_writer(db_name: Optional[str]):
        db_key = db_name or "unknown"
        safe = _sanitize_filename(db_key)
        path = os.path.join(OUTPUT_DIR, f"{safe}.jsonl")
        if safe not in writers:
            writers[safe] = open(path, "w", encoding="utf-8")  # overwrites per run
            counts[safe] = 0
        return writers[safe], safe

    try:
        for urn in iter_dataset_urns():
            try:
                rows = fetch_dataset_columns(urn)
                for row in rows:
                    writer, key = get_writer(row.get("database"))
                    writer.write(json.dumps(row, ensure_ascii=False) + "\n")
                    counts[key] += 1
            except Exception as e:
                # If dataset fails, write to a generic 'unknown.jsonl'
                writer, key = get_writer("unknown")
                writer.write(json.dumps({"error": f"{urn}: {e}"}, ensure_ascii=False) + "\n")
                counts[key] += 1
    finally:
        for fh in writers.values():
            try:
                fh.close()
            except Exception:
                pass

    # Summary
    print(f"Exported per database to: {os.path.abspath(OUTPUT_DIR)}")
    for db_file, n in sorted(counts.items()):
        print(f"- {db_file}.jsonl: {n} rows")

if __name__ == "__main__":
    main()

Exported per database to: /app/datahub_metadata
- classicmodels.jsonl: 59 rows
